# Notebook - Phase 6

## Triage of the 11 problematic tickers

In [ ]:
import io
from xml.etree.ElementTree import tostring

import yaml
from pandas.core.frame import DataFrame
from sympy.polys.dispersion import dispersionset

with open("/Users/hugo/quant-research/config/equity_universes.yaml", 'r') as f:
    config = yaml.safe_load(f)
universe = 'russell1000'

tickers = config[universe]
print(len(tickers))

In [ ]:
import pandas as pd
df = pd.read_csv("/Users/hugo/quant-research/config/russell1000_sectors.csv")
df.head()

In [ ]:
# check if symbols match
df_tolist = df["Symbol"].to_list()
print(set(df_tolist)-set(tickers))
print(set(tickers)-set(df_tolist))

In [ ]:
def normalize(x):
    return x.strip().replace("."," ")

In [ ]:
df["Symbol"] = df["Symbol"].apply(normalize)
print(set(df["Symbol"].tolist())-set(tickers))

Creation of the `.csv` file where status/rename/industry/exchange/delist info live

First create a DataFrame and export to csv. Columns:

`conID | ticker | secIdType | secId | sec_type | primary_exchange | currency | gics_sector | status | delist_date | former_ticker | last_fetch`

In [ ]:
import nest_asyncio; nest_asyncio.apply()
from src.execution.ib_connection import IBConnection
from ib_async import Stock
conn = IBConnection()
conn.connect()

print(f"Connected: {conn.is_connected()}")
print(f"Accounts: {conn.ib.managedAccounts()}")

cds = conn.ib.reqContractDetails(Stock('AAPL', 'SMART', 'USD'))
print(len(cds), cds[0].contract.conId, cds[0].contract.primaryExchange)

conn.disconnect()

In [ ]:
import nest_asyncio; nest_asyncio.apply()      # needed for ib_async in Jupyter
from ib_async import Stock
from src.execution.ib_connection import IBConnection

with IBConnection(client_id=1) as ib:           # host/port from .env
    cds = ib.reqContractDetails(Stock('AAPL', 'SMART', 'USD'))
    print(len(cds), cds[0].contract.conId, cds[0].contract.primaryExchange)

## Chat phase 6a. universe
The new strategy is to build on iShares IWB daily holdings. The ETF replicates the Russell1000 and therefore provides all its constituents.

In [ ]:
import pandas as pd

path = '/Users/hugo/quant-research/config/IWB_holdings.csv'
raw = pd.read_csv(path,skiprows=9,thousands=',',na_values=['-'])
raw.head()

In [ ]:
# Inspect data
raw.shape, raw.dtypes

In [ ]:
# Parse date of holdings

from datetime import date, datetime
from itertools import islice
from pathlib import Path

AS_OF_LABEL = "Fund Holdings as of"
N_HEADER_LINES = 9


def parse_as_of_date(path: str | Path) -> date:
    """Return the holdings 'as of' date from the iShares CSV metadata block.

    Raises ValueError if the label is missing or the date can't be parsed.
    """
    with open(path, encoding="utf-8") as f:
        for line in islice(f, N_HEADER_LINES):
            if line.startswith(AS_OF_LABEL):
                raw = line.split(",", maxsplit=1)[1].strip().strip('"')
                return datetime.strptime(raw, "%b %d, %Y").date()
    raise ValueError(f"'{AS_OF_LABEL}' not found in first {N_HEADER_LINES} lines of {path}")

In [ ]:
parse_as_of_date(path)

In [ ]:
asof = parse_as_of_date(path)


In [ ]:
# to check that it doesn't work and prints an understandable error
path2 = '/Users/hugo/quant-research/config/IWB_holdings_3.csv'
parse_as_of_date(path2)

In [ ]:
float_vars = raw.dtypes[raw.dtypes == 'float64'].index.tolist()

In [ ]:
(raw[float_vars].astype(str) == '-').any()

In [ ]:
raw.loc[raw['Ticker'] == 'NVDA', 'Weight (%)']

In [ ]:
print(raw['Weight (%)'].sum())

It sums to more than 100% because the fund contains also cash and a couple of other assets

In [ ]:
# Step 4
holdings = raw[raw['Asset Class'] == 'Equity']
print(holdings.shape)

In [ ]:
raw[raw['Ticker'].isin(['HOLX','UWMC-R-W'])]

In [ ]:
exchanges = holdings["Exchange"].unique().tolist()
print(exchanges)

In [ ]:
exclude_noexchange = exchanges[4]

In [ ]:
# creating an allowing list for tickers: uppercase letters plus a space and a class letter
# Using regex
holdings = holdings[holdings['Ticker'].str.fullmatch(r'[A-Z]+(.[A-Z])?')].copy()
print(holdings.shape)

In [ ]:
# above takes care only of non-usual tickers name. need to take care of those tickers whose exchanges are not the traditional exchanges, should be automatic


In [ ]:
holdings = raw[raw['Asset Class'] == 'Equity']
holdings = holdings[holdings['Ticker'].str.fullmatch(r'[A-Z]+(.[A-Z])?')].copy()
exchanges = holdings["Exchange"].unique().tolist()
exclude_exchange = [exchanges[4]]
holdings = holdings.loc[~holdings['Exchange'].isin(exclude_exchange)].copy()

In [ ]:
# create dictionary to rename and select columns
var_select = {'Ticker':'ticker', 'Name':'name', 'Sector':'sector','Market Value':'market_value','Price':'price','Exchange':'exchange'}
holdings = holdings.loc[:,var_select.keys()].rename(columns=var_select).copy()
holdings.head()

In [ ]:
# check that tickers are unique in the final data
print(holdings.ticker.is_unique)

In [ ]:
# solution
TICKER_PATTERN = r"[A-Z]+( [A-Z])?"          # common share, optional class letter: BRK B
LISTED_EXCHANGES = {"NYSE", "NASDAQ", "Cboe BZX", "Nyse Mkt Llc"}
COLUMNS = {"Ticker": "ticker", "Name": "name", "Sector": "sector",
           "Market Value": "market_value", "Price": "price",
           "Exchange": "exchange", "Location": "location"}

equities = raw[raw["Asset Class"] == "Equity"]

valid_ticker = equities["Ticker"].str.fullmatch(TICKER_PATTERN)
listed = equities["Exchange"].isin(LISTED_EXCHANGES)
keep = valid_ticker & listed

dropped = equities.loc[~keep, ["Ticker", "Name", "Exchange"]]
print(f"Dropped {len(dropped)} equity rows:\n{dropped.to_string(index=False)}")

holdings = equities.loc[keep, list(COLUMNS)].rename(columns=COLUMNS)
print(holdings.shape, holdings.ticker.is_unique)

In [ ]:
# Step 5
from datetime import timedelta

MAX_ROWS = 1040
MIN_ROWS = 990
SECTORS = raw.Sector.unique().tolist()
SECTORS.remove('Cash and/or Derivatives')
MAX_AGE_DAYS = 7

def validate_holdings(holdings: pd.DataFrame, as_of: date, today: date):
    """
    Validates the holdings
    """
    failures = []
    if holdings.shape[0]<MIN_ROWS or holdings.shape[0]>MAX_ROWS:
        failures.append(f"Truncated download: {holdings.shape[0]} rows in DataFrame.")

    if not holdings.ticker.is_unique:
        duplicated_tickers = holdings.loc[holdings['tickers'].duplicated(),'ticker'].unique().tolist()
        failures.append(f"Duplicated tickers: {duplicated_tickers}")

    if holdings[["ticker","name","sector","price","market_value"]].isna().any().any():
        failures.append(f"Missing values recorded")

    if not ((holdings.price>0).all() and (holdings.market_value>0).all()):
        failures.append(f"Null or negative price(s)/market value(s) recorded")

    if holdings.sector.unique().tolist() not in SECTORS:
        failures.append(f"Sector not recognized")

    if as_of>today or (as_of<today-timedelta(days=7)):
        failures.append(f"Today or tomorrow not recognized")


    raise ValueError(failure for failure in failures)

In [ ]:
(holdings.price>0).all()

In [ ]:
raw.Sector.unique()

In [ ]:
# Solution step 5
from datetime import date

MIN_ROWS, MAX_ROWS = 990, 1040
MAX_AGE_DAYS = 7
REQUIRED_COLUMNS = ["ticker", "name", "sector", "price", "market_value"]
GICS_SECTORS = {
    "Communication", "Consumer Discretionary", "Consumer Staples", "Energy",
    "Financials", "Health Care", "Industrials", "Information Technology",
    "Materials", "Real Estate", "Utilities",
}


def validate_holdings(holdings: pd.DataFrame, as_of: date, today: date) -> None:
    """Check the cleaned IWB holdings are plausible before they become a snapshot.

    Raises ValueError listing every failed check.
    """
    failures = []

    n = len(holdings)
    if not MIN_ROWS <= n <= MAX_ROWS:
        failures.append(f"row count {n} outside [{MIN_ROWS}, {MAX_ROWS}]")

    dupes = holdings.loc[holdings["ticker"].duplicated(), "ticker"].unique().tolist()
    if dupes:
        failures.append(f"duplicate tickers: {dupes}")

    missing = holdings[REQUIRED_COLUMNS].isna().sum()
    missing = missing[missing > 0]
    if not missing.empty:
        failures.append(f"missing values: {missing.to_dict()}")

    bad = holdings.loc[(holdings["price"] <= 0) | (holdings["market_value"] <= 0), "ticker"].tolist()
    if bad:
        failures.append(f"non-positive price or market value: {bad}")

    unknown = set(holdings["sector"].dropna()) - GICS_SECTORS
    if unknown:
        failures.append(f"unknown sectors: {sorted(unknown)}")

    age = (today - as_of).days
    if not 0 <= age <= MAX_AGE_DAYS:
        failures.append(f"as_of {as_of} is {age} days before today {today} (allowed 0-{MAX_AGE_DAYS})")

    if failures:
        raise ValueError("Holdings validation failed:\n- " + "\n- ".join(failures))

In [ ]:
# Step 6
import requests

IWB_HOLDINGS_URL = (
    "https://www.blackrock.com/varnish-api/blk-one01-product-data/product-data/api/v1/"
    "get-fund-document?appType=PRODUCT_PAGE&appSubType=ISHARES&targetSite=us-ishares"
    "&locale=en_US&portfolioId=239707&userType=individual&component=holdings"
)

response = requests.get(IWB_HOLDINGS_URL, timeout=30)

In [ ]:
print(response.status_code) # 200 means success

In [ ]:
print(repr(response.text[:500]))

In [ ]:
header_pos = response.text.find("Ticker")
n_lines_top = response.text[:header_pos].count("\n")

In [ ]:
footer_pos = header_pos + response.text[header_pos:].find("\n\n")

In [ ]:
print(response.content[:100])

In [ ]:
response.raise_for_status()

In [ ]:
header_pos = response.text.find("Ticker")
for line in response.text[:header_pos].splitlines():
    if line.startswith(AS_OF_LABEL):
        raw = line.split(",", maxsplit=1)[1].strip().strip('"')
        as_of = datetime.strptime(raw, "%b %d, %Y").date()

print(as_of)

In [ ]:
from pathlib import Path
raw_bytes_path = Path("data/production/universe/raw")
raw_bytes_path.mkdir(parents=True, exist_ok=True)
file_path = raw_bytes_path / f"IWB_holdings_{as_of}.csv"
file_path.write_bytes(response.content)

In [ ]:
footer_text = response.text[footer_pos:]
print(len(footer_text.splitlines()))

In [ ]:
import re
import io
from pathlib import Path
from datetime import date, datetime
import requests

IWB_HOLDINGS_URL = (
    "https://www.blackrock.com/varnish-api/blk-one01-product-data/product-data/api/v1/"
    "get-fund-document?appType=PRODUCT_PAGE&appSubType=ISHARES&targetSite=us-ishares"
    "&locale=en_US&portfolioId=239707&userType=individual&component=holdings"
)

AS_OF_LABEL = "Fund Holdings as of"
TICKER_PATTERN = r"[A-Z]+( [A-Z])?"          # common share, optional class letter: BRK B
LISTED_EXCHANGES = {"NYSE", "NASDAQ", "Cboe BZX", "Nyse Mkt Llc"}
COLUMNS = {"Ticker": "ticker", "Name": "name", "Sector": "sector",
           "Market Value": "market_value", "Price": "price",
           "Exchange": "exchange", "Location": "location"}
MIN_ROWS, MAX_ROWS = 990, 1040
MAX_AGE_DAYS = 7
REQUIRED_COLUMNS = ["ticker", "name", "sector", "price", "market_value"]
GICS_SECTORS = {
    "Communication", "Consumer Discretionary", "Consumer Staples", "Energy",
    "Financials", "Health Care", "Industrials", "Information Technology",
    "Materials", "Real Estate", "Utilities",
}

def parse_as_of_date(meta_text: str) -> date:
    """
    Returns the holdings 'as of' date from the text metadata block.
    Raises ValueError if the label is missing or the date can't be parsed.
    """
    for line in meta_text.splitlines():
        if line.startswith(AS_OF_LABEL):
            raw = line.split(",", maxsplit=1)[1].strip().strip('"')
            return datetime.strptime(raw, "%b %d, %Y").date()
    raise ValueError(f"'{AS_OF_LABEL}' not found in text metadata")

def clean_holdings_df(df: pd.DataFrame) -> pd.DataFrame:
    equities = df[df["Asset Class"] == "Equity"]
    valid_ticker = equities["Ticker"].str.fullmatch(TICKER_PATTERN)
    listed = equities["Exchange"].isin(LISTED_EXCHANGES)
    keep = valid_ticker & listed
    dropped = equities.loc[~keep, ["Ticker", "Name", "Exchange"]]
    print(f"Dropped {len(dropped)} equity rows:\n{dropped.to_string(index=False)}")
    return equities.loc[keep, list(COLUMNS)].rename(columns=COLUMNS)

response = requests.get(IWB_HOLDINGS_URL, timeout=30)
response.raise_for_status()

# Finding table's header and footer positions
header_pos = response.text.find("Ticker")
match = re.search(r"\r?\n\r?\n", response.text[header_pos:])
footer_pos = header_pos + match.start() if match else len(response.text)

# Parsing the date
as_of = parse_as_of_date(response.text[:header_pos])

# Save the raw bytes
raw_bytes_path = Path("config/universe/raw_bytes")
raw_bytes_path.mkdir(parents=True, exist_ok=True)
file_path = raw_bytes_path / f"IWB_holdings_{as_of}.csv"
file_path.write_bytes(response.content)

# Parse the table
table_text = response.text[header_pos:footer_pos]
raw = pd.read_csv(io.StringIO(table_text), thousands=",", na_values="-")

holdings = clean_holdings_df(raw)

validate_holdings(holdings,as_of,today=date.today())


In [ ]:
from src.data_pipelines.universe.ishares_holdings import load_latest_holdings
as_of, holdings = load_latest_holdings()

In [ ]:
from pathlib import Path
import logging
from datetime import date
from src.paths import UNIVERSE_RAW_DIR
from src.data_pipelines.universe.ishares_holdings import holdings_from_text

logging.basicConfig(level=logging.INFO)

path = sorted(UNIVERSE_RAW_DIR.glob("IWB_holdings_*.csv"))[-1]   # latest file
as_of, holdings = holdings_from_text(path.read_text(encoding="utf-8"), today=date.today())

In [ ]:
print(as_of, holdings.shape)

In [ ]:
holdings.tail()